## 1. Data upload and package installation

**Requested prompt sections:** Data upload and package versions; the annual estimates, pairwise group comparisons, and graphs requested below.

**Code explanation:** Imports the pinned packages, reports versions, and loads the supplied Excel workbook. In Colab, the cell prompts for the workbook; locally, it reads `data/NAVCO2-1_ForPublication.xlsx`. It sets the colorblind-friendly palette and saves future figures in `output_ba65064/`.

**Assumptions:** The workbook has the required year, campaign ID, fatality, success, flank-response, violent-flank, and campaign-size columns. The optional `%pip` installation line is commented out; uncomment it to install the pinned dependencies.

In [ ]:
# To install the exact pinned packages from this repository, uncomment and run:
# %pip install -r requirements.txt

import pathlib
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from IPython.display import display

# Limited colorblind-friendly palette, kept consistent for each category.
INK = "#3A3532"
MUTED = "#746A64"
PAPER = "#FAF7F3"
GRID = "#E7DCD5"
RUST = "#B24C2F"
BLUE = "#4C6572"
RUST_PALE = "#D9B9AA"
RUST_DARK = "#704034"
PERIOD_PALETTE = {"1994 or before": BLUE, "After 1994": RUST}
LABEL_PALETTE = {
    "Not successful": BLUE, "Successful": RUST,
    "1994 or before": BLUE, "After 1994": RUST,
    "No violent flank": BLUE, "Nonviolent discipline": "#E69F00",
    "Toleration of violent flank": "#009E73", "Internal disagreement": "#CC79A7",
    "Primarily violent campaign": RUST, "Violent flank": "#56B4E9",
    "1–999": BLUE, "1,000–9,999": "#E69F00", "10,000–99,999": "#009E73",
    "100,000–499,999": "#CC79A7", "500,000–1 million": "#56B4E9",
    "At least 1 million": RUST,
}

sns.set_theme(style="whitegrid", rc={
    "font.family": "serif", "font.serif": ["DejaVu Serif", "Georgia", "serif"],
    "figure.facecolor": PAPER, "axes.facecolor": PAPER,
    "axes.edgecolor": GRID, "axes.labelcolor": INK,
    "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 15,
    "axes.labelsize": 10, "font.size": 9,
})

DATA_PATH = pathlib.Path("data/NAVCO2-1_ForPublication.xlsx")
OUTPUT_DIR = pathlib.Path("output_ba65064")
OUTPUT_DIR.mkdir(exist_ok=True)
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        DATA_PATH = pathlib.Path(next(iter(uploaded)))
except ImportError:
    pass

df_raw = pd.read_excel(DATA_PATH)
package_names = ["pandas", "numpy", "scipy", "matplotlib", "seaborn", "openpyxl"]
package_versions = pd.DataFrame({
    "Package": package_names,
    "Version": [importlib.metadata.version(name) for name in package_names]
})
print(f"Loaded {df_raw.shape[0]:,} campaign-year rows across {df_raw['id'].nunique():,} campaign IDs")
print(f"Workbook: {DATA_PATH}")
display(package_versions)


## 2. Cleaning

**Requested prompt section:** Replace `-99` with NaN.

**Code explanation:** Copies the imported workbook, replaces numeric and text `-99` values with missing values, and converts analysis variables to numeric types. The original dataframe remains available as `df_raw`.

**Assumptions:** `-99` means unknown or unavailable, and blank spreadsheet cells are missing. Each analysis excludes rows missing the needed outcome or grouping variable.

In [ ]:
df = df_raw.copy()
df = df.replace([-99, "-99"], np.nan)
for column in ["year", "id", "fatalities_high", "success", "flank_response", "violent_flank", "camp_size"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print(f"-99 values remaining: {int((df == -99).sum().sum())}")
print(f"Missing fatalities_high after cleaning: {int(df['fatalities_high'].isna().sum())}")
display(df[["id", "year", "fatalities_high", "success", "flank_response", "violent_flank", "camp_size"]].head())


## 3. Analysis by year

**Requested prompt sections:** Show estimates of log-adjusted `fatalities_high` (“Peak Fatality Estimate”) by year and success group with confidence intervals, without a regression; compare 1994 or before with after 1994; report the two means and a two-tailed test of H0: mean(before) − mean(after) = 0 at α = 0.05; create separate log-adjusted box and violin figures.

**Code explanation:** The over-time line chart shows annual means of `log1p(fatalities_high)` by final success, with 95% bootstrap confidence intervals and no fitted regression. The pre/post comparison tests raw fatality means using a two-sided Welch t-test. The box and violin charts display `log1p` values.

**Assumptions:** 1994 is included in the earlier period; post-1994 starts in 1995. The Welch test treats campaign-year rows as independent although campaigns may appear in several years. `success` is final campaign success (0 = not successful; 1 = successful). Line-chart intervals use bootstrap resampling within year and success group. `log1p(x) = log(1+x)` retains zero-fatality observations.

**Visual key:** Success groups use consistent colors, and the legend is to the right. The violin shows quartile lines, white-circle means, and jittered translucent individual campaign-year points.

In [ ]:
# Annual estimates by success with 95% bootstrap confidence intervals; no regression.
time_data = df.dropna(subset=["year", "fatalities_high", "success"]).copy()
time_data["log_adjusted_fatalities"] = np.log1p(time_data["fatalities_high"])
time_data["Outcome"] = time_data["success"].map({0: "Not successful", 1: "Successful"})
time_data["Outcome"] = pd.Categorical(time_data["Outcome"], categories=["Not successful", "Successful"], ordered=True)

fig, ax = plt.subplots(figsize=(10.5, 6.2))
sns.lineplot(data=time_data, x="year", y="log_adjusted_fatalities", hue="Outcome",
             hue_order=["Not successful", "Successful"], palette=LABEL_PALETTE,
             estimator="mean", errorbar=("ci", 95), n_boot=1000, seed=42,
             marker="o", linewidth=2, ax=ax)

fig.suptitle("Annual fatality estimates", x=0.08, y=0.99, ha="left", fontsize=18, fontweight="bold")
fig.text(0.08, 0.90, "Mean log-adjusted peak fatality estimate by year and final success; bars show 95% CIs",
         ha="left", color=MUTED, fontsize=9)
ax.set_xlabel("Campaign year")
ax.set_ylabel("log(1 + Peak Fatality Estimate)")
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
ax.grid(axis="x", visible=False)
fig.text(0.08, 0.025, "Source: NAVCO 2.1. Points summarize campaign-year observations; no regression is fit.", color=MUTED, fontsize=8)
fig.subplots_adjust(left=0.10, right=0.76, bottom=0.12, top=0.82)
fig.savefig(OUTPUT_DIR / "01_annual_estimates_ci.png", dpi=180, bbox_inches="tight", facecolor=PAPER)
plt.show()

# Compare raw fatalities by the requested year grouping and test the difference in means.
year_data = df.dropna(subset=["year", "fatalities_high"]).copy()
year_data["Period"] = np.where(year_data["year"] <= 1994, "1994 or before", "After 1994")
period_order = ["1994 or before", "After 1994"]
values_before = year_data.loc[year_data["Period"] == "1994 or before", "fatalities_high"]
values_after = year_data.loc[year_data["Period"] == "After 1994", "fatalities_high"]
avFatal_1994_and_before = values_before.mean()
avFatal_post_1994 = values_after.mean()
period_summary = year_data.groupby("Period")["fatalities_high"].agg(["count", "mean", "median", "std"]).reindex(period_order)
display(period_summary)
print(f"avFatal_1994_and_before = {avFatal_1994_and_before:,.3f}")
print(f"avFatal_post_1994 = {avFatal_post_1994:,.3f}")
print(f"Difference (before − after) = {avFatal_1994_and_before - avFatal_post_1994:,.3f}")
alpha = 0.05
welch_result = stats.ttest_ind(values_before, values_after, equal_var=False,
                               alternative="two-sided", nan_policy="omit")
print(f"Welch two-tailed test, H0: avFatal_1994_and_before − avFatal_post_1994 = 0; "
      f"t={welch_result.statistic:.4f}, df={welch_result.df:.2f}, p={welch_result.pvalue:.6g}, alpha={alpha}")
if welch_result.pvalue < alpha:
    print("Decision: reject H0 at alpha = 0.05.")
else:
    print("Decision: fail to reject H0 at alpha = 0.05.")

year_data["log_adjusted_fatalities"] = np.log1p(year_data["fatalities_high"])
fig, ax = plt.subplots(figsize=(8.2, 5.4))
sns.boxplot(data=year_data, x="Period", y="log_adjusted_fatalities", order=period_order,
            width=0.48, hue="Period", hue_order=period_order, palette=PERIOD_PALETTE,
            dodge=False, legend=False, linewidth=1.0, ax=ax)
year_box_legend = [Patch(facecolor=PERIOD_PALETTE[p], label=p) for p in period_order]
ax.legend(handles=year_box_legend, frameon=False, loc="center left",
          bbox_to_anchor=(1.02, 0.5), fontsize=8)
fig.suptitle("Fatality estimates by period", x=0.09, y=0.99, ha="left", fontsize=17, fontweight="bold")
fig.text(0.09, 0.90, "Campaign-years grouped at 1994; outcome shown as log(1 + estimate)",
         ha="left", color=MUTED, fontsize=9)
ax.set_xlabel("")
ax.set_ylabel("log(1 + Peak Fatality Estimate)")
ax.grid(axis="x", visible=False)
fig.text(0.09, 0.025, "Source: NAVCO 2.1. Box plot uses campaign-year observations.", color=MUTED, fontsize=8)
fig.subplots_adjust(left=0.12, right=0.77, bottom=0.13, top=0.82)
fig.savefig(OUTPUT_DIR / "02_year_boxplot.png", dpi=180, bbox_inches="tight", facecolor=PAPER)
plt.show()

fig, ax = plt.subplots(figsize=(8.2, 5.4))
sns.violinplot(data=year_data, x="Period", y="log_adjusted_fatalities", order=period_order,
               hue="Period", hue_order=period_order, palette=PERIOD_PALETTE,
               dodge=False, legend=False, inner="quartile", cut=0, linewidth=1.0, ax=ax)
for violin_body in ax.collections:
    if isinstance(violin_body, PolyCollection):
        violin_body.set_alpha(0.42)
sns.stripplot(data=year_data, x="Period", y="log_adjusted_fatalities", order=period_order,
              hue="Period", hue_order=period_order, palette=PERIOD_PALETTE, dodge=False,
              alpha=0.58, size=2.6, jitter=0.14, legend=False, ax=ax)
year_violin_means = year_data.groupby("Period")["log_adjusted_fatalities"].mean().reindex(period_order)
ax.scatter([0, 1], year_violin_means.values, marker="o", s=62, facecolor="white",
           edgecolor=INK, linewidth=1.0, zorder=5, label="Mean")
fig.suptitle("Fatality estimates by period", x=0.09, y=0.99, ha="left", fontsize=17, fontweight="bold")
fig.text(0.09, 0.90, "Quartile lines; white circles mark means; translucent points are individual campaign-years",
         ha="left", color=MUTED, fontsize=9)
ax.set_xlabel("")
ax.set_ylabel("log(1 + Peak Fatality Estimate)")
ax.grid(axis="x", visible=False)
year_legend = [Patch(facecolor=PERIOD_PALETTE[p], label=p) for p in period_order]
year_legend.append(Line2D([0], [0], marker="o", linestyle="", markerfacecolor="white",
                          markeredgecolor=INK, label="Mean"))
ax.legend(handles=year_legend, frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
fig.text(0.09, 0.025, "Source: NAVCO 2.1. Each point is one campaign-year.", color=MUTED, fontsize=8)
fig.subplots_adjust(left=0.12, right=0.77, bottom=0.13, top=0.82)
fig.savefig(OUTPUT_DIR / "03_year_violin.png", dpi=180, bbox_inches="tight", facecolor=PAPER)
plt.show()


## 4. Analysis by group and pairwise comparisons

**Requested prompt sections:** Compare campaign-level fatality estimates across flank-response, violent-flank, and campaign-size subgroups; use descriptive category labels; create log-adjusted violin plots; retain the focused violent-flank code 1 (no violent flank) versus code 2 (violent flank) plot; report all subgroup pairwise comparisons with first-group mean, mean difference, two-tailed p-value, and Bonferroni-adjusted p-value. Add line charts of pairwise estimates and confidence intervals.

**Code explanation:** Annual fatalities are averaged within campaign ID. Each campaign receives its modal valid category for each grouping variable; ties go to the lowest code. Two-sided unequal-variance Welch tests compare the raw campaign-level means for every category pair. Bonferroni adjustment is applied separately within each grouping family. The pairwise line charts show raw mean differences and 95% Welch confidence intervals; the violins show log-adjusted campaign outcomes, quartiles, white-circle means, and jittered points. The comparison table is displayed and saved as an image.

**Assumptions:** `-99` and blanks are missing. Each campaign contributes one summary observation per grouping; campaigns are treated as independent. Pairwise comparisons are two-tailed and use the raw fatality scale. Violin plots use `log1p` for display. Confidence intervals describe uncertainty in mean differences and do not imply causation.

In [ ]:
# One record per campaign: average observed annual fatalities and modal valid categories.
campaign_data = df.dropna(subset=["id", "fatalities_high"]).groupby("id", as_index=False).agg(
    fatalities_high=("fatalities_high", "mean")
)
for column in ["flank_response", "violent_flank", "camp_size"]:
    category_counts = (df.dropna(subset=["id", column])
                         .groupby(["id", column]).size()
                         .reset_index(name="year_count")
                         .sort_values(["id", "year_count", column], ascending=[True, False, True]))
    category_modes = category_counts.drop_duplicates("id").set_index("id")[column]
    campaign_data[column] = campaign_data["id"].map(category_modes)

category_specs = {
    "flank_response": {
        "title": "Fatality estimates by flank response",
        "short_title": "Flank response",
        "labels": {0: "No violent flank", 1: "Nonviolent discipline", 2: "Toleration of violent flank", 3: "Internal disagreement"}
    },
    "violent_flank": {
        "title": "Fatality estimates by violent-flank status",
        "short_title": "Violent-flank status",
        "labels": {0: "Primarily violent campaign", 1: "No violent flank", 2: "Violent flank"}
    },
    "camp_size": {
        "title": "Fatality estimates by campaign size",
        "short_title": "Campaign size",
        "labels": {0: "1–999", 1: "1,000–9,999", 2: "10,000–99,999",
                   3: "100,000–499,999", 4: "500,000–1 million", 5: "At least 1 million"}
    }
}

pairwise_rows = []
for column, spec in category_specs.items():
    analysis = campaign_data.dropna(subset=[column, "fatalities_high"]).copy()
    analysis[column] = analysis[column].astype(int)
    observed_codes = sorted(analysis[column].unique())
    family_rows = []

    # Pairwise Welch tests compare raw campaign-level means; intervals estimate mean differences.
    for i, code_a in enumerate(observed_codes):
        values_a = analysis.loc[analysis[column] == code_a, "fatalities_high"]
        for code_b in observed_codes[i + 1:]:
            values_b = analysis.loc[analysis[column] == code_b, "fatalities_high"]
            pair_test = stats.ttest_ind(values_a, values_b, equal_var=False,
                                        alternative="two-sided", nan_policy="omit")
            mean_difference = values_a.mean() - values_b.mean()
            variance_a = values_a.var(ddof=1)
            variance_b = values_b.var(ddof=1)
            variance_term_a = variance_a / len(values_a)
            variance_term_b = variance_b / len(values_b)
            standard_error_difference = np.sqrt(variance_term_a + variance_term_b)
            welch_df = ((variance_term_a + variance_term_b) ** 2 /
                        (variance_term_a ** 2 / (len(values_a) - 1) +
                         variance_term_b ** 2 / (len(values_b) - 1)))
            critical_t = stats.t.ppf(0.975, welch_df)
            label_a = spec["labels"].get(code_a, str(code_a))
            label_b = spec["labels"].get(code_b, str(code_b))
            family_rows.append({
                "Factor": column,
                "Comparison": f"{spec['short_title']}: {label_a} vs {label_b}",
                "Mean fatalities (first group)": values_a.mean(),
                "Difference in means (first − second)": mean_difference,
                "95% CI lower": mean_difference - critical_t * standard_error_difference,
                "95% CI upper": mean_difference + critical_t * standard_error_difference,
                "Two-tailed p-value": pair_test.pvalue
            })

    number_of_family_tests = len(family_rows)
    for row in family_rows:
        row["Bonferroni-adjusted p-value"] = min(row["Two-tailed p-value"] * number_of_family_tests, 1.0)
        pairwise_rows.append(row)

    # Grouped violins show log-adjusted distributions, quartiles, means, and individual campaigns.
    analysis["log_adjusted_fatalities"] = np.log1p(analysis["fatalities_high"])
    analysis["Category"] = analysis[column].map(spec["labels"])
    plot_order = [spec["labels"].get(code, str(code)) for code in observed_codes]
    fig, ax = plt.subplots(figsize=(max(8, len(plot_order) * 2.1), 5.8))
    sns.violinplot(data=analysis, x="Category", y="log_adjusted_fatalities", order=plot_order,
                   hue="Category", hue_order=plot_order,
                   palette={label: LABEL_PALETTE[label] for label in plot_order},
                   dodge=False, legend=False, inner="quartile", cut=0, linewidth=1.0, ax=ax)
    for violin_body in ax.collections:
        if isinstance(violin_body, PolyCollection):
            violin_body.set_alpha(0.42)
    sns.stripplot(data=analysis, x="Category", y="log_adjusted_fatalities", order=plot_order,
                  hue="Category", hue_order=plot_order,
                  palette={label: LABEL_PALETTE[label] for label in plot_order}, dodge=False,
                  alpha=0.62, size=3.0, jitter=0.13, legend=False, ax=ax)
    category_means = analysis.groupby("Category")["log_adjusted_fatalities"].mean().reindex(plot_order)
    ax.scatter(np.arange(len(plot_order)), category_means.values, marker="o", s=62,
               facecolor="white", edgecolor=INK, linewidth=1.0, zorder=5, label="Mean")
    fig.suptitle(spec["title"], x=0.08, y=0.99, ha="left", fontsize=17, fontweight="bold")
    fig.text(0.08, 0.90, f"{len(analysis)} campaigns; quartiles shown; white circles mark means",
             ha="left", color=MUTED, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("log(1 + campaign-average Peak Fatality Estimate)")
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="x", visible=False)
    group_legend = [Patch(facecolor=LABEL_PALETTE[label], label=label) for label in plot_order]
    group_legend.append(Line2D([0], [0], marker="o", linestyle="", markerfacecolor="white",
                               markeredgecolor=INK, label="Mean"))
    ax.legend(handles=group_legend, frameon=False, loc="center left",
              bbox_to_anchor=(1.02, 0.5), fontsize=8)
    fig.text(0.08, 0.025, "Source: NAVCO 2.1. Points are translucent jittered campaign-level observations.", color=MUTED, fontsize=8)
    fig.subplots_adjust(left=0.10, right=0.70, bottom=0.17, top=0.82)
    fig.savefig(OUTPUT_DIR / f"04_{column}_violin.png", dpi=180, bbox_inches="tight", facecolor=PAPER)
    plt.show()

# Focused comparison: campaigns whose modal code is 1 (no flank) or 2 (violent flank).
flank_focus = campaign_data.loc[campaign_data["violent_flank"].isin([1, 2])].copy()
flank_focus["Flank group"] = flank_focus["violent_flank"].map({1: "No violent flank", 2: "Violent flank"})
flank_focus["log_adjusted_fatalities"] = np.log1p(flank_focus["fatalities_high"])
focus_order = ["No violent flank", "Violent flank"]
focus_means = flank_focus.groupby("Flank group")["fatalities_high"].mean().reindex(focus_order)
fig, ax = plt.subplots(figsize=(7.8, 5.5))
sns.violinplot(data=flank_focus, x="Flank group", y="log_adjusted_fatalities", order=focus_order,
               hue="Flank group", hue_order=focus_order, dodge=False, legend=False,
               palette={focus_order[0]: BLUE, focus_order[1]: RUST},
               inner="quartile", cut=0, linewidth=1.0, ax=ax)
for violin_body in ax.collections:
    if isinstance(violin_body, PolyCollection):
        violin_body.set_alpha(0.42)
sns.stripplot(data=flank_focus, x="Flank group", y="log_adjusted_fatalities", order=focus_order,
              hue="Flank group", hue_order=focus_order,
              palette={label: LABEL_PALETTE[label] for label in focus_order}, dodge=False,
              alpha=0.62, size=3.0, jitter=0.13, legend=False, ax=ax)
focus_log_means = flank_focus.groupby("Flank group")["log_adjusted_fatalities"].mean().reindex(focus_order)
ax.scatter([0, 1], focus_log_means.values, marker="o", s=62, facecolor="white",
           edgecolor=INK, linewidth=1.0, zorder=5, label="Mean")
fig.suptitle("Campaign fatalities: no flank vs violent flank", x=0.09, y=0.99,
             ha="left", fontsize=16, fontweight="bold")
fig.text(0.09, 0.90, f"Campaign-level averages; codes 1 and 2 only; quartiles shown; n={len(flank_focus)}",
         ha="left", color=MUTED, fontsize=9)
ax.set_xlabel("")
ax.set_ylabel("log(1 + campaign-average Peak Fatality Estimate)")
ax.grid(axis="x", visible=False)
focus_legend = [Patch(facecolor=LABEL_PALETTE[label], label=label) for label in focus_order]
focus_legend.append(Line2D([0], [0], marker="o", linestyle="", markerfacecolor="white",
                           markeredgecolor=INK, label="Mean"))
ax.legend(handles=focus_legend, frameon=False, loc="center left",
          bbox_to_anchor=(1.02, 0.5), fontsize=8)
fig.text(0.09, 0.025, "Raw-scale campaign means: " +
         f"no violent flank = {focus_means.iloc[0]:,.1f}; violent flank = {focus_means.iloc[1]:,.1f}",
         color=MUTED, fontsize=8)
fig.subplots_adjust(left=0.12, right=0.73, bottom=0.14, top=0.82)
fig.savefig(OUTPUT_DIR / "05_no_violent_flank_vs_violent_flank.png", dpi=180,
            bbox_inches="tight", facecolor=PAPER)
plt.show()

# Pairwise table with original requested columns plus confidence intervals for the difference estimates.
pairwise_table = pd.DataFrame(pairwise_rows)
pairwise_display = pairwise_table.drop(columns=["Factor"])
print("Pairwise Welch tests; Bonferroni correction is within each grouping family.")
display(pairwise_display.round(6))

table_rows = []
for _, row in pairwise_display.iterrows():
    table_rows.append([
        row["Comparison"],
        f"{row['Mean fatalities (first group)']:,.1f}",
        f"{row['Difference in means (first − second)']:,.1f}",
        f"{row['95% CI lower']:,.1f}",
        f"{row['95% CI upper']:,.1f}",
        f"{row['Two-tailed p-value']:.4g}",
        f"{row['Bonferroni-adjusted p-value']:.4g}"
    ])
fig, ax = plt.subplots(figsize=(19, 10.5))
ax.axis("off")
fig.suptitle("Pairwise comparisons of campaign fatality estimates", x=0.04, y=0.985,
             ha="left", fontsize=17, fontweight="bold")
fig.text(0.04, 0.91, "Two-sided Welch tests; Bonferroni adjusted within each factor family; 95% CIs for mean differences",
         ha="left", color=MUTED, fontsize=9)
table = ax.table(cellText=table_rows,
                 colLabels=["Comparison", "First-group mean", "Mean difference",
                            "95% CI lower", "95% CI upper", "Two-tailed p", "Bonferroni p"],
                 colWidths=[0.31, 0.12, 0.12, 0.11, 0.11, 0.10, 0.13],
                 cellLoc="left", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(7.5)
table.scale(1, 1.32)
for (row_index, column_index), cell in table.get_celld().items():
    cell.set_edgecolor(GRID)
    if row_index == 0:
        cell.set_facecolor(RUST)
        cell.set_text_props(color="white", weight="bold")
    elif row_index % 2 == 0:
        cell.set_facecolor("#F1E8E2")
        cell.set_text_props(color=INK)
    else:
        cell.set_facecolor(PAPER)
        cell.set_text_props(color=INK)
fig.subplots_adjust(left=0.03, right=0.98, top=0.87, bottom=0.04)
pairwise_image_path = OUTPUT_DIR / "06_pairwise_comparisons.png"
fig.savefig(pairwise_image_path, dpi=180, bbox_inches="tight", facecolor=PAPER)
plt.show()

# Line charts show each pairwise mean-difference estimate with its 95% Welch confidence interval.
for column, spec in category_specs.items():
    family_plot = pairwise_table.loc[pairwise_table["Factor"] == column].copy()
    comparison_labels = [label.replace(f"{spec['short_title']}: ", "") for label in family_plot["Comparison"]]
    estimates = family_plot["Difference in means (first − second)"].to_numpy()
    lower_bounds = family_plot["95% CI lower"].to_numpy()
    upper_bounds = family_plot["95% CI upper"].to_numpy()
    x_positions = np.arange(len(family_plot))

    fig, ax = plt.subplots(figsize=(max(9, len(family_plot) * 1.25), 6.2))
    ax.errorbar(x_positions, estimates,
                yerr=[estimates - lower_bounds, upper_bounds - estimates],
                fmt="o-", color=BLUE, ecolor=RUST, capsize=3, linewidth=1.4,
                markersize=5, label="Mean difference with 95% CI")
    ax.axhline(0, color=INK, linewidth=1.0, linestyle="--", label="No mean difference")
    fig.suptitle(f"Pairwise estimates: {spec['short_title']}", x=0.08, y=0.99,
                 ha="left", fontsize=17, fontweight="bold")
    fig.text(0.08, 0.90, "Mean difference (first minus second); vertical bars show 95% Welch confidence intervals",
             ha="left", color=MUTED, fontsize=9)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(comparison_labels, rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("Difference in mean fatalities")
    ax.set_xlabel("Pairwise comparison")
    ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    ax.grid(axis="x", visible=False)
    fig.subplots_adjust(left=0.10, right=0.78, bottom=0.34, top=0.82)
    fig.savefig(OUTPUT_DIR / f"07_{column}_pairwise_ci.png",
                dpi=180, bbox_inches="tight", facecolor=PAPER)
    plt.show()
